# KALIA — Data Prep (CPU)

Turns TinyStories + FineWeb-Edu into token files (`train.bin`, `val.bin`).

**Notebook settings**
- Accelerator: **None** (all CPU work; CPU quota is unlimited)
- Internet: **On**

**When it finishes:** `Save Version` (run in background) → open the version → *Output* → *Create Dataset*, name it `kalia-tokens`, keep it **private**.

In [ ]:
!pip install -q tiktoken datasets

In [ ]:
GITHUB_REPO = "https://github.com/CHANGE_ME/kalia.git"  # <-- edit this

!git clone {GITHUB_REPO} || (cd kalia && git pull)
%cd kalia

In [ ]:
!mkdir -p /kaggle/working/data
!python prepare.py --source tinystories --out /kaggle/working/data/train_stories.bin --max-tokens 500000000 --meta /kaggle/working/data/meta.json
!python prepare.py --source fineweb --out /kaggle/working/data/train_fineweb.bin --max-tokens 2000000000 --meta /kaggle/working/data/meta.json
!cat /kaggle/working/data/meta.json

In [ ]:
from pathlib import Path

data = Path("/kaggle/working/data")
stories = (data / "train_stories.bin").read_bytes()
fineweb = (data / "train_fineweb.bin").read_bytes()

# 5M tokens from each source held out for validation (uint16: 2 bytes per token)
val = stories[:10_000_000] + fineweb[:10_000_000]
train = stories[10_000_000:] + fineweb[10_000_000:]

(data / "val.bin").write_bytes(val)
(data / "train.bin").write_bytes(train)
(data / "train_stories.bin").unlink()
(data / "train_fineweb.bin").unlink()

print("train tokens:", (data / "train.bin").stat().st_size // 2)
print("val tokens:", (data / "val.bin").stat().st_size // 2)

Done. Now `Save Version` → *Create Dataset* `kalia-tokens` (private), then open **kalia-train**.